In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import os, glob

In [ ]:
import matplotlib as mpl
from matplotlib import font_manager
# Custom font path.
font_path = '../../data/Arial.ttf'

# Add the font to matplotlib.
font_manager.fontManager.addfont(font_path)

# Get the registered font name.
custom_font = font_manager.FontProperties(fname=font_path)
font_name = custom_font.get_name()
# Set the global font.
mpl.rcParams['font.family'] = font_name

In [ ]:
country = 'Nigeria'
files = glob.glob(f'../../data/processed/{country}/*/labels_norm.pkl')
len(files)

In [ ]:
targets_df = pd.read_csv(f'../../data/processed/0labels/{country}.csv')
targets_df

In [ ]:
df_dict = {}
for file in files:
    city = file.split('/')[-2]
    print(city)
    df = pd.read_pickle(file)
    df_dict[city] = df

In [ ]:
# targets = list(set(targets_df['ID'].to_list()) & set(df_dict['JO'].columns))
targets = targets_df['ID'].to_list()
targets

In [ ]:
# fig, axes = plt.subplots(3, 6, figsize=(18, 8))
# fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig, axes = plt.subplots(1, 3, figsize=(18, 3))
# fig, axes = plt.subplots(1, 5, figsize=(15, 3))
# fig, axes = plt.subplots(1, 6, figsize=(18, 3))
# fig, axes = plt.subplots(4, 6, figsize=(18, 12))

cities = list(df_dict.keys())
colormap = plt.cm.get_cmap('tab20', len(cities))
colors = [colormap(i) for i in range(len(cities))]

for city, df in df_dict.items():
    for i, target in enumerate(targets):
        if target not in df.columns:
            continue
        # ax =axes[i//6, i%6]
        ax = axes[i]
        sns.kdeplot(df[target], ax=ax, color=colors[cities.index(city)], label=city, legend=False)
        if df[target].dropna().shape[0] == 0:
            axes[i//6, i%6].set_visible(False)
            continue
        ax.set_title(targets_df[targets_df['ID'] == target]['Code'].values[0].replace('\\', ''), fontsize=16)

        ax.set_ylabel('KDE Density')
        ax.set_xlabel('')
        if i % 6 != 0:
            ax.set_ylabel('')

# custom legend
# handles, labels = axes[0,0].get_legend_handles_labels()
# for i in range(1, len(axes[1])):
#     axes[1, i].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
# axes[3, 4].axis('off')
# axes[3, 5].axis('off')
# fig.legend(handles, labels, frameon=False, loc='lower right', ncol=1, bbox_to_anchor=(0.8, 0.1), fontsize=14)
fig.legend(handles, labels, frameon=False, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.1), fontsize=14)


plt.tight_layout()
plt.show()

## Spatial distribution per indicator.

In [ ]:
from esda.moran import Moran
from libpysal.weights import Queen

In [ ]:
labels = gpd.GeoDataFrame(labels, geometry='geometry')

In [ ]:
# Build a Queen-contiguity weights matrix.
w = Queen.from_dataframe(labels)
w.transform = 'r'  # row-standardize

# Detect and drop isolated units.
if w.islands:
    print(f"Found isolated units: {w.islands}")
    # Boolean mask: keep only non-isolated units.
    mask = ~labels.index.isin(w.islands)
    city_geo = labels[mask]
    # Rebuild the weights matrix to match the filtered data.
    w = Queen.from_dataframe(labels)
    w.transform = 'r'  # row-standardize again

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 8))

for i, target in enumerate(targets):
    if labels[target].dropna().shape[0] == 0:
        axes[i//6, i%6].set_visible(False)
        continue
    labels.plot(column=target, ax=axes[i//6, i%6])
    # Compute Moran's I.
    y = labels[target]
    # Drop NaNs automatically.
    y_clean = y.dropna()
    # Confirm there are enough valid rows left.
    if y_clean.empty or y_clean.nunique() == 1:
        print(f"Skipping {target}: insufficient valid data.")
        continue
    # Rebuild data and weights.
    df_clean = labels.loc[y_clean.index]  # keep rows that have valid data
    w_clean = Queen.from_dataframe(df_clean)
    w_clean.transform = 'r'
    # Compute Moran's I.
    try:
        moran = Moran(df_clean[target], w_clean)
        # print(f"{target}: Moran's I = {moran.I:.4f}, p-value = {moran.p_sim:.4f}")
    except Exception as e:
        print(f"Error processing {target}: {e}")
    
    axes[i//6, i%6].set_title(f"{target}\n{moran.I:.2f}, {moran.p_sim:.2f}")

plt.suptitle(f'Spatial Distribution of {city}-{country} labels')
plt.tight_layout()
plt.savefig(f'../../data/processed/{country}/{city}/label_distribution_spatial.png')
plt.show()

## Count images per spatial unit.

In [ ]:
paths = pd.read_pickle(paths_path)
paths.head()

In [ ]:
paths.groupby('GEOID').size().describe()

In [ ]:
plt.figure(figsize=(10, 6))
paths.groupby('GEOID').size().hist(bins=100)
plt.text(
    0.5, 0.5, 
    f'Mean: {paths.groupby("GEOID").size().mean():.2f}\nStd: {paths.groupby("GEOID").size().std():.2f}\nMin: {paths.groupby("GEOID").size().min()}\nMax: {paths.groupby("GEOID").size().max()}', 
    transform=plt.gca().transAxes
)
plt.title(f'Distribution of number of paths per units in {city}-{country}')
plt.savefig(f'../../data/processed/{country}/{city}/path_distribution.png')
plt.show()

In [ ]:
labels = labels.merge(paths.groupby('GEOID').size().rename('num_paths'), on='GEOID')

In [ ]:
labels.plot(column='num_paths', legend=True)
plt.title(f'Number of images per units in {city}-{country}')
plt.savefig(f'../../data/processed/{country}/{city}/path_distribution_spatial.png')
plt.show()